# Camada Silver

**Bases deste notebook**
- `pesquisa_satisfacao.csv`
- `custos.csv`
- `canais.csv`

# 1. Configuração

In [0]:
from pyspark.sql import functions as F

catalog = "vcredit"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Origem: {bronze_schema}")
print(f"Destino: {silver_schema}")

# 2. Limpeza dos dados

## 2.1. Pesquisa de satisfação
**Tratamentos realizados: remoção de espaços, conversão da nota para inteiro, validação das notas entre 1 e 5 e identificação de duplicidades.**

In [0]:
df_pesquisa_bronze = spark.table(f"{bronze_schema}.pesquisa_satisfacao")
df_pesquisa = (
    df_pesquisa_bronze
    .withColumn(
        "id_pesquisa",
        F.when(
            F.trim(F.col("id_pesquisa").cast("string")) == "",
            None
        ).otherwise(F.trim(F.col("id_pesquisa").cast("string")))
    )
    .withColumn(
        "id_chamado",
        F.when(
            F.trim(F.col("id_chamado").cast("string")) == "",
            None
        ).otherwise(F.trim(F.col("id_chamado").cast("string")))
    )
    .withColumn(
        "nota_atendimento",
        F.expr("try_cast(NULLIF(TRIM(nota_atendimento), 'NULL') AS INT)")
    )
    .withColumn(
        "nota_valida",
        F.col("nota_atendimento").between(1, 5)
    )
    .withColumn(
        "id_pesquisa_duplicado",
        F.col("id_pesquisa").isNotNull() &
        (F.count("id_pesquisa").over(
            __import__("pyspark").sql.Window.partitionBy("id_pesquisa")
        ) > 1)
    )
)

df_pesquisa = df_pesquisa.dropDuplicates()
(
    df_pesquisa.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.pesquisa_satisfacao")
)

display(df_pesquisa.limit(10))

## 2.2. Custos
**Tratamentos realizados: remoção de símbolos monetários e textos desnecessários, padronização dos separadores decimais, conversão para DECIMAL(12,2), preservação do valor original e validação de custos não negativos.**

In [0]:
df_custos_bronze = spark.table(f"{bronze_schema}.custos")
df_custos = (
    df_custos_bronze
    .withColumn(
        "id_custo",
        F.trim(F.col("id_custo").cast("string"))
    )
    .withColumn(
        "id_chamado",
        F.trim(F.col("id_chamado").cast("string"))
    )
    .withColumn(
        "custo_original",
        F.col("custo").cast("string")
    )
)

valor_limpo = F.lower(F.trim(F.col("custo_original")))
valor_limpo = F.regexp_replace(valor_limpo, r"r\$", "")
valor_limpo = F.regexp_replace(valor_limpo, r"reais?", "")
valor_limpo = F.regexp_replace(valor_limpo, r"\s+", "")
valor_limpo = F.regexp_replace(valor_limpo, r"[^0-9,.\-]", "")

tem_virgula = F.instr(valor_limpo, ",") > 0
tem_ponto = F.instr(valor_limpo, ".") > 0
virgula_depois_ponto = (F.instr(valor_limpo, ",") > F.instr(valor_limpo, "."))

valor_normalizado = (
    F.when(
        tem_virgula & tem_ponto & virgula_depois_ponto,
        F.regexp_replace(
            F.regexp_replace(valor_limpo, r"\.", ""),
            ",",
            "."
        )
    )
    .when(
        tem_virgula & tem_ponto & (~virgula_depois_ponto),
        F.regexp_replace(valor_limpo, ",", "")
    )
    .when(
        tem_virgula,
        F.regexp_replace(valor_limpo, ",", ".")
    )
    .otherwise(valor_limpo)
)
df_custos = (
    df_custos
    .withColumn(
        "_custo_normalizado",
        valor_normalizado
    )
    .withColumn(
        "custo",
        F.col("_custo_normalizado").cast("decimal(12,2)")
    )
    .withColumn(
        "custo_valido",
        F.col("custo").isNotNull() &
        (F.col("custo") >= F.lit(0))
    )
    .drop("_custo_normalizado")
    .dropDuplicates()
)
(
    df_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.custos")
)

display(df_custos.limit(10))

## 2.3. Canais
**Tratamentos realizados: remoção de espaços, padronização dos textos em letras minúsculas, validação dos campos obrigatórios e remoção de registros duplicados exatos.**

In [0]:
df_canais_bronze = spark.table(
    f"{bronze_schema}.canais"
)

df_canais = (
    df_canais_bronze
    .withColumn(
        "id_canal",
        F.trim(F.col("id_canal").cast("string"))
    )
    .withColumn(
        "nome_canal",
        F.lower(F.trim(F.col("nome_canal").cast("string")))
    )
    .withColumn(
        "tipo_canal",
        F.lower(F.trim(F.col("tipo_canal").cast("string")))
    )
    .withColumn(
        "status_canal",
        F.lower(F.trim(F.col("status_canal").cast("string")))
    )
    .withColumn(
        "canal_valido",
        F.col("id_canal").isNotNull() &
        (F.col("id_canal") != "") &
        F.col("nome_canal").isNotNull() &
        (F.col("nome_canal") != "") &
        F.col("tipo_canal").isNotNull() &
        (F.col("tipo_canal") != "") &
        F.col("status_canal").isNotNull() &
        (F.col("status_canal") != "")
    )
    .dropDuplicates()
)

(
    df_canais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.canais")
)

display(df_canais)

#3. Relatório de Qualidade

In [0]:
def gerar_resumo(nome_tabela, df, chave, coluna_valida=None):
    total = df.count()

    chaves_nulas = df.filter(
        F.col(chave).isNull() |
        (F.trim(F.col(chave).cast("string")) == "")
    ).count()

    resultado = {
        "tabela": nome_tabela,
        "total_registros": total,
        "chaves_nulas": chaves_nulas
    }

    if coluna_valida:
        invalidos = df.filter(
            ~F.coalesce(F.col(coluna_valida), F.lit(False))
        ).count()
        resultado["registros_invalidos"] = invalidos
    return resultado


df_silver_pesquisa = spark.table(f"{silver_schema}.pesquisa_satisfacao")
df_silver_custos = spark.table(f"{silver_schema}.custos")

resumo = [
    gerar_resumo(
        "pesquisa_satisfacao",
        df_silver_pesquisa,
        "id_pesquisa",
        "nota_valida"
    ),
    gerar_resumo(
        "custos",
        df_silver_custos,
        "id_custo",
        "custo_valido"
    )
]

display(spark.createDataFrame(resumo))